# 단계 3 — YOLOv8n-cls Fine-Tuning (연기 탐지)

복강경 수술 영상에서 연기를 탐지하는 이진 분류 모델을 학습한다.

| 항목 | 내용 |
|---|---|
| 모델 | YOLOv8n-cls (ImageNet pretrained → fine-tuning) |
| 목표 | **Recall ≥ 0.99** (연기를 놓치는 것이 가장 치명적) |
| 데이터 | `data/smoke_cls/` — train 769장×2, val 192장×2 |
| 클래스 | `smoke` (gt 없음) / `no_smoke` (gt 붙음) |

## 0. 환경 확인

In [6]:
import torch
from pathlib import Path

# GPU 사용 가능 여부 확인
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스  : {device}")
if device == "cuda":
    print(f"GPU 이름       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 데이터셋 경로 및 이미지 수 확인
BASE_DIR  = Path("..").resolve()
DATA_DIR  = BASE_DIR / "data" / "smoke_cls"

for split in ("train", "val"):
    for cls in ("smoke", "no_smoke"):
        count = len(list((DATA_DIR / split / cls).glob("*.png")))
        print(f"  {split:5s}/{cls:10s} : {count}장")

사용 디바이스  : cuda
GPU 이름       : NVIDIA GeForce RTX 4080 SUPER
VRAM           : 17.2 GB
  train/smoke      : 769장
  train/no_smoke   : 769장
  val  /smoke      : 192장
  val  /no_smoke   : 192장


## 1. Fine-Tuning 학습

**핵심 파라미터 설명:**
- `epochs=50` + `patience=10` → 10 epoch 동안 개선 없으면 조기 종료
- `imgsz=224` → YOLOv8n-cls 기본 입력 크기
- `batch=32` → GPU VRAM에 따라 조정 가능
- `device=0` → GPU 사용 (CPU면 `device='cpu'`로 변경)
- `workers=2` → **Windows에서 DataLoader 프로세스 수 제한 (기본 8 → CPU 과부하 방지)**

> 학습 완료 후 `runs/smoke_detector/yolov8n_cls/weights/best.pt` 에 최적 가중치 저장됨

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

results = model.train(
    data      = str(DATA_DIR),
    epochs    = 50,
    imgsz     = 224,
    batch     = 32,
    device    = 0 if torch.cuda.is_available() else "cpu",
    patience  = 10,
    workers   = 0,
    cache     = "ram",
    project   = str(BASE_DIR / "runs" / "smoke_detector"),
    name      = "yolov8n_cls",
    exist_ok  = True,

    # ── 조명 불변성 확보를 위한 극단적 Augmentation ──────
    # 목표: 모델이 "절대 밝기" 대신 연기 고유의 텍스처/산란 패턴을 학습
    #
    # hsv_v=0.9: 밝기를 원본의 10%~190% 범위로 무작위 변경
    #   → 어두운 핀포인트 조명(mean≈90)부터 밝은 수술 영상(mean≈130)까지
    #     모두 학습 범위에 포함되어 밝기로는 smoke/no_smoke를 구분 못하게 함
    hsv_v     = 0.9,
    # hsv_s=0.7: 채도 변화로 연기의 탁한 색감 다양성 흡수
    hsv_s     = 0.7,
    # fliplr/flipud: 수술 방향/카메라 각도 무관하게 일반화
    fliplr    = 0.5,
    flipud    = 0.3,
    # translate/scale: 카메라 거리/위치 차이 흡수
    translate = 0.1,
    scale     = 0.3,
)

## 2. Val 세트 평가 및 Confusion Matrix

학습된 best.pt로 val 전체를 평가하고 클래스별 Recall을 확인한다.

In [8]:
import time
import random
from pathlib import Path

BEST_PT   = BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "weights" / "best.pt"
CONF_THRES = 0.3   # pretrained_test.py와 동일한 threshold 유지

print(f"[로드] {BEST_PT}")
best_model = YOLO(str(BEST_PT))

# val 전체 이미지로 평가
val_smoke_imgs    = list((DATA_DIR / "val" / "smoke").glob("*.png"))
val_no_smoke_imgs = list((DATA_DIR / "val" / "no_smoke").glob("*.png"))

tp = fp = fn = tn = 0
total_ms = 0.0

# smoke 이미지 추론
for img in val_smoke_imgs:
    t0   = time.perf_counter()
    res  = best_model.predict(str(img), verbose=False)
    total_ms += (time.perf_counter() - t0) * 1000

    names     = res[0].names                          # {0: 'no_smoke', 1: 'smoke'}
    top1_idx  = int(res[0].probs.top1)
    pred_name = names[top1_idx]

    if pred_name == "smoke":
        tp += 1
    else:
        fn += 1   # ← 가장 위험한 오류 (연기 놓침)

# no_smoke 이미지 추론
for img in val_no_smoke_imgs:
    t0   = time.perf_counter()
    res  = best_model.predict(str(img), verbose=False)
    total_ms += (time.perf_counter() - t0) * 1000

    names     = res[0].names
    top1_idx  = int(res[0].probs.top1)
    pred_name = names[top1_idx]

    if pred_name == "no_smoke":
        tn += 1
    else:
        fp += 1

# 지표 계산
n_total   = tp + fp + fn + tn
accuracy  = (tp + tn) / n_total
precision = tp / max(tp + fp, 1)
recall    = tp / max(tp + fn, 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-9)
avg_ms    = total_ms / n_total

print("\n" + "=" * 55)
print("  Fine-tuned YOLOv8n-cls 평가 결과 (val 전체)")
print("=" * 55)
print(f"  샘플 수        : smoke {len(val_smoke_imgs)}장, no_smoke {len(val_no_smoke_imgs)}장")
print(f"  conf threshold : {CONF_THRES}")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Accuracy       : {accuracy:.4f}")
print(f"  Precision      : {precision:.4f}")
print(f"  Recall (smoke) : {recall:.4f}  ← 핵심 지표  (목표 ≥ 0.99)")
print(f"  F1 Score       : {f1:.4f}")
print(f"  평균 추론 속도 : {avg_ms:.1f} ms/프레임")
print("=" * 55)

[로드] F:\일\Capstone_Project\AI_Capstone_Project\runs\smoke_detector\yolov8n_cls\weights\best.pt

  Fine-tuned YOLOv8n-cls 평가 결과 (val 전체)
  샘플 수        : smoke 192장, no_smoke 192장
  conf threshold : 0.3
  TP=187  FP=6  FN=5  TN=186
  Accuracy       : 0.9714
  Precision      : 0.9689
  Recall (smoke) : 0.9740  ← 핵심 지표  (목표 ≥ 0.99)
  F1 Score       : 0.9714
  평균 추론 속도 : 8.0 ms/프레임


## 3. Confusion Matrix 시각화

In [9]:
import matplotlib.pyplot as plt
import numpy as np

cm = np.array([[tp, fn],
               [fp, tn]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax)

classes = ["smoke (Positive)", "no_smoke (Negative)"]
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: smoke", "Pred: no_smoke"], fontsize=11)
ax.set_yticks([0, 1]); ax.set_yticklabels(["GT: smoke", "GT: no_smoke"],     fontsize=11)

# 셀 값 표시
for i in range(2):
    for j in range(2):
        label = ["TP", "FN", "FP", "TN"][i * 2 + j]
        ax.text(j, i, f"{label}\n{cm[i,j]}", ha="center", va="center",
                fontsize=13, color="white" if cm[i,j] > cm.max() / 2 else "black")

ax.set_title(f"Confusion Matrix\nRecall(smoke)={recall:.4f}  F1={f1:.4f}", fontsize=12)
plt.tight_layout()
plt.savefig(str(BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "confusion_matrix_custom.png"), dpi=150)
plt.show()
print("저장 완료")

<Figure size 500x400 with 2 Axes>

저장 완료


## 4. 목표 달성 판단 및 재학습 가이드

- **Recall ≥ 0.99** 달성 시 → 완료
- 미달 시 → conf threshold를 낮추거나 재학습 진행

In [10]:
TARGET_RECALL = 0.99

print("=" * 55)
if recall >= TARGET_RECALL:
    print(f"  [달성] Recall {recall:.4f} ≥ {TARGET_RECALL} — 목표 충족!")
    print(f"  최종 모델: {BEST_PT}")
else:
    gap = TARGET_RECALL - recall
    print(f"  [미달] Recall {recall:.4f} < {TARGET_RECALL} (부족분 {gap:.4f})")
    print()
    print("  ── 개선 옵션 ──────────────────────────────────")
    print(f"  옵션 A: conf threshold 낮추기 (현재 {CONF_THRES} → 0.1 시도)")
    print("  옵션 B: epochs 늘리기 (50 → 100)")
    print("  옵션 C: 클래스 가중치 추가 (pos_weight 설정)")
    print("  ──────────────────────────────────────────────")
print("=" * 55)

  [미달] Recall 0.9740 < 0.99 (부족분 0.0160)

  ── 개선 옵션 ──────────────────────────────────
  옵션 A: conf threshold 낮추기 (현재 0.3 → 0.1 시도)
  옵션 B: epochs 늘리기 (50 → 100)
  옵션 C: 클래스 가중치 추가 (pos_weight 설정)
  ──────────────────────────────────────────────
